# NB09 — Base-Paper Head-to-Head (Hoque & Seddiqui 2025)

Faithful replication of the base paper's protocol on **its own data**: the **facebook_44001** source with
its **native 5 classes** (Not Bully, Sexual, Troll, Religious, Threat), **70/15/15** split. We apply
**our full framework** (FGM + R-Drop + EMA + MSD + focal + sampler) and the weighted ensemble, then
report on the 15% test and compare to the base paper.

**Base-paper targets:** macro-F1 = **89.23** · per-class F1 → Not Bully 91.51 · Sexual 88.45 ·
Troll 84.46 · Religious 93.74 · Threat 75.79.

No leakage: trained only on the facebook-70% train; within-facebook dedup prevents train/test overlap.


In [ ]:
import os, re, json, time, math, random, warnings
import numpy as np, pandas as pd, unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score, classification_report
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
DEBUG = False; DEBUG_SAMPLES = 500
CFG = {"models": {
       "banglishbert": "csebuetnlp/banglishbert",
       "banglabert": "csebuetnlp/banglabert",
       "muril": "google/muril-base-cased",
       "xlmr": "xlm-roberta-base"},
       "max_length":128,"batch_size":16,"eval_batch_size":32,"grad_accum_steps":2,"num_workers":0,
       "epochs":8,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
       "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
       "dropout":0.25,"head_hidden_dim":384,"n_msd":4,"patience":3,"use_fp16":torch.cuda.is_available(),
       "use_balanced_sampler":True,"sampler_alpha":0.5,"use_fgm":True,"fgm_epsilon":1.0,
       "use_rdrop":True,"rdrop_alpha":0.5,"use_ema":True,"ema_decay":0.999}
RUN_MODELS = ["banglishbert", "banglabert", "muril", "xlmr"]; SEEDS=[42]
RAW="../data/merged/benchmark_raw.csv"; OUT="../outputs/basepaper"; os.makedirs(OUT,exist_ok=True)
SPLIT_SEED=42; TEXT="text_clean"; LABEL="label5b"
BASE_PAPER={"macro_f1":0.8923,"per_class":{"Not Bully":0.9151,"Sexual":0.8845,"Troll":0.8446,"Religious":0.9374,"Threat":0.7579}}
if DEBUG: CFG["epochs"]=2; SEEDS=[42]; print("⚠ DEBUG")

In [ ]:
# ── Load facebook, preprocess, dedup-within, native 5 classes, 70/15/15 ─────
df=pd.read_csv(RAW); fb=df[df["source"]=="facebook_44001"].reset_index(drop=True).copy()
print(f"facebook rows: {len(fb):,}")
URL=re.compile(r"https?://\S+|www\.\S+"); MEN=re.compile(r"@[\w]+"); HASH=re.compile(r"#(\S+)")
ZW=re.compile(r"[\u200b\u200c\u200d\ufeff]"); WS=re.compile(r"\s+"); ELONG=re.compile(r"(.)\1{2,}")
EMO=re.compile("["+"\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF"
               "\U0001F1E0-\U0001F1FF\U00002700-\U000027BF\U0001F900-\U0001F9FF\U00002600-\U000026FF]+",flags=re.UNICODE)
def clean(t):
    if not isinstance(t,str): return ""
    t=unicodedata.normalize("NFKC",t); t=ZW.sub("",t); t=URL.sub(" [URL] ",t); t=MEN.sub(" [USER] ",t)
    t=HASH.sub(r" [HASHTAG] \1 ",t); t=EMO.sub(" [EMOJI] ",t); t=ELONG.sub(r"\1\1",t); return WS.sub(" ",t).strip()
fb["text_clean"]=fb["text"].apply(clean); fb=fb[fb["text_clean"].str.len()>0]
n0=len(fb); fb=fb.drop_duplicates("text_clean").reset_index(drop=True)
print(f"within-facebook dedup: {n0:,} → {len(fb):,}")

# native labels → base-paper 5-class names
NAME={"not bully":"Not Bully","sexual":"Sexual","troll":"Troll","religious":"Religious","threat":"Threat"}
fb[LABEL]=fb["label_type"].astype(str).str.strip().str.lower().map(NAME)
fb=fb[fb[LABEL].notna()].reset_index(drop=True)
print("\n5-class distribution:"); print(fb[LABEL].value_counts().to_string())
assert fb[LABEL].nunique()==5

tr,tmp=train_test_split(fb,test_size=0.30,random_state=SPLIT_SEED,stratify=fb[LABEL])
va,te=train_test_split(tmp,test_size=0.50,random_state=SPLIT_SEED,stratify=tmp[LABEL])
if DEBUG:
    tr=tr.groupby(LABEL,group_keys=False).apply(lambda g:g.sample(min(len(g),DEBUG_SAMPLES//5),random_state=42))
    va=va.sample(min(len(va),300),random_state=42); te=te.sample(min(len(te),300),random_state=42)
classes=sorted(fb[LABEL].unique()); enc={c:i for i,c in enumerate(classes)}; NC=5
print(f"\nsplit 70/15/15  train={len(tr):,} val={len(va):,} test={len(te):,} | enc={enc}")
y_val=va[LABEL].map(enc).values; y_test=te[LABEL].map(enc).values

In [ ]:
# ── Training machinery (same framework as NB05) ─────────────────────────────
class DS(Dataset):
    def __init__(s,df,tok,ml): s.t=df.reset_index(drop=True)[TEXT].fillna("").astype(str).tolist(); s.y=[int(enc.get(v,-1)) for v in df[LABEL]]; s.tok,s.ml=tok,ml
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=s.tok(s.t[i],max_length=s.ml,truncation=True,padding=False); it={k:torch.tensor(v,dtype=torch.long) for k,v in e.items()}; it["label"]=torch.tensor(s.y[i],dtype=torch.long); return it
class Coll:
    def __init__(s,tok): s.tok=tok
    def __call__(s,fs):
        lb=torch.stack([f["label"] for f in fs]); ft=[{k:v for k,v in f.items() if k!="label"} for f in fs]; b=s.tok.pad(ft,padding=True,return_tensors="pt"); b["label"]=lb; return b
class Focal(nn.Module):
    def __init__(s,g=2.0,w=None,ls=0.0): super().__init__(); s.g,s.w,s.ls=g,w,ls
    def forward(s,lg,t): ce=F.cross_entropy(lg,t,weight=s.w,reduction="none",label_smoothing=s.ls); return (((1-torch.exp(-ce))**s.g)*ce).mean()
def cw_(series,beta=0.999,mw=10.0):
    m=series.map(enc).dropna().astype(int); c=m.value_counts().sort_index(); w=np.ones(NC,dtype=np.float32)
    for i in range(NC):
        k=int(c.get(i,0))
        if k>0: w[i]=1.0/max((1.0-beta**k)/max(1.0-beta,1e-12),1e-9)
    w=np.clip(w,w.min(),w.min()*mw); w=w/w.mean(); return torch.tensor(w,dtype=torch.float32,device=device)
def sampler_(df,alpha=0.5):
    y=df[LABEL].map(enc).fillna(-1).astype(int).values; cc=np.bincount(y[y>=0],minlength=NC).astype(float); cc[cc==0]=1.0
    cw=(1.0/cc)**alpha; sw=np.where(y>=0,cw[np.clip(y,0,None)],0.0); return WeightedRandomSampler(torch.as_tensor(sw,dtype=torch.double),len(sw),replacement=True)
class Head(nn.Module):
    def __init__(s,h,n,dp=0.25,inner=384,nm=4):
        super().__init__(); inner=min(inner,h); s.pre=nn.Sequential(nn.Linear(h,inner),nn.GELU(),nn.LayerNorm(inner)); s.dr=nn.ModuleList([nn.Dropout(dp) for _ in range(nm)]); s.o=nn.Linear(inner,n)
    def forward(s,x):
        h=s.pre(x)
        if s.training and len(s.dr)>1: return torch.stack([s.o(d(h)) for d in s.dr],0).mean(0)
        return s.o(s.dr[0](h))
class Enc(nn.Module):
    def __init__(s,name,n,dp=0.25,inner=384,nm=4):
        super().__init__(); s.encoder=AutoModel.from_pretrained(name); h=s.encoder.config.hidden_size
        s._tti=s.encoder.config.model_type.lower() not in ("roberta","xlm-roberta"); s.head=Head(h,n,dp,inner,nm)
    def _pool(s,o,m): hs=o.last_hidden_state; return 0.5*hs[:,0]+0.5*((hs*m.unsqueeze(-1).float()).sum(1)/m.sum(1,keepdim=True).float().clamp(1))
    def forward(s,ids,mask,tti=None):
        kw={"input_ids":ids,"attention_mask":mask}
        if s._tti and tti is not None: kw["token_type_ids"]=tti
        return s.head(s._pool(s.encoder(**kw),mask))
def up(model,e,h,wd):
    nd=["bias","LayerNorm.weight","LayerNorm.bias"]; g=[]
    for grp,lr in [(model.encoder,e),(model.head,h)]:
        d,nde=[],[]
        for n,p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g+=[{"params":d,"lr":lr,"weight_decay":wd},{"params":nde,"lr":lr,"weight_decay":0.0}]
    return g
class FGM:
    def __init__(s,m,eps=1.0): s.m,s.e,s.k,s.b=m,eps,"word_embeddings",{}
    def attack(s):
        for n,p in s.m.named_parameters():
            if p.requires_grad and s.k in n and p.grad is not None:
                s.b[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(s.e*p.grad/nm)
    def restore(s):
        for n,p in s.m.named_parameters():
            if n in s.b: p.data=s.b[n]
        s.b={}
class EMA:
    def __init__(s,m,d=0.999): s.d=d; s.s={n:p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}; s.b={}
    def update(s,m):
        for n,p in m.named_parameters():
            if p.requires_grad: s.s[n].mul_(s.d).add_(p.detach(),alpha=1-s.d)
    def apply_shadow(s,m):
        s.b={n:p.detach().clone() for n,p in m.named_parameters() if p.requires_grad}
        for n,p in m.named_parameters():
            if p.requires_grad: p.data.copy_(s.s[n])
    def restore(s,m):
        for n,p in m.named_parameters():
            if p.requires_grad and n in s.b: p.data.copy_(s.b[n])
        s.b={}
def rkl(lp,lq):
    pl,ql=F.log_softmax(lp,-1),F.log_softmax(lq,-1); p,q=pl.exp(),ql.exp(); return 0.5*((p*(pl-ql)).sum(-1)+(q*(ql-pl)).sum(-1)).mean()
def seed_(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def logits_(model,loader):
    model.eval(); o=[]
    for b in loader:
        b={k:v.to(device) for k,v in b.items()}; o.append(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).cpu())
    return torch.cat(o)
print("machinery ok")

In [ ]:
# ── Train the encoders, collect val/test logits ─────────────────────────────
def train_model(mk,name,seed):
    seed_(seed)
    _bs  = CFG.get("batch_size_override",{}).get(mk, CFG["batch_size"])
    _acc = CFG["grad_accum_steps"] * (CFG["batch_size"] // _bs) if _bs != CFG["batch_size"] else CFG["grad_accum_steps"]
    tok=AutoTokenizer.from_pretrained(name); coll=Coll(tok)
    lkw=dict(collate_fn=coll,num_workers=0,pin_memory=torch.cuda.is_available())
    if CFG["use_balanced_sampler"]:
        tl=DataLoader(DS(tr,tok,CFG["max_length"]),batch_size=_bs,sampler=sampler_(tr,CFG["sampler_alpha"]),shuffle=False,drop_last=True,**lkw)
    else:
        tl=DataLoader(DS(tr,tok,CFG["max_length"]),batch_size=_bs,shuffle=True,drop_last=True,**lkw)
    vl=DataLoader(DS(va,tok,CFG["max_length"]),batch_size=CFG["eval_batch_size"],shuffle=False,**lkw)
    tel=DataLoader(DS(te,tok,CFG["max_length"]),batch_size=CFG["eval_batch_size"],shuffle=False,**lkw)
    model=Enc(name,NC,CFG["dropout"],CFG["head_hidden_dim"],CFG["n_msd"]).to(device)
    focal=Focal(CFG["focal_gamma"],cw_(tr[LABEL],CFG["class_weight_beta"]),CFG["label_smoothing"])
    opt=torch.optim.AdamW(up(model,CFG["encoder_lr"],CFG["head_lr"],CFG["weight_decay"]))
    ns=math.ceil(len(tl)/_acc)*CFG["epochs"]; sch=get_linear_schedule_with_warmup(opt,int(ns*CFG["warmup_ratio"]),ns)
    scaler=torch.amp.GradScaler("cuda") if (CFG["use_fp16"] and device.type=="cuda") else None
    fgm=FGM(model,CFG["fgm_epsilon"]) if CFG["use_fgm"] else None; ema=EMA(model,CFG["ema_decay"]) if CFG["use_ema"] else None
    @torch.no_grad()
    def vmac():
        model.eval(); P,Y=[],[]
        for b in vl:
            b={k:v.to(device) for k,v in b.items()}; P.extend(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).argmax(-1).cpu().numpy()); Y.extend(b["label"].cpu().numpy())
        Y=np.array(Y); m=Y>=0; return f1_score(Y[m],np.array(P)[m],average="macro",zero_division=0)
    best,noimp=-1.0,0; sd=f"{OUT}/{mk}_seed{seed}"; os.makedirs(sd,exist_ok=True)
    for ep in range(CFG["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step,b in enumerate(tl,1):
            b={k:v.to(device) for k,v in b.items()}; y=b["label"]
            with torch.autocast(device_type=device.type,enabled=scaler is not None):
                l1=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                if CFG["use_rdrop"]:
                    l2=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                    loss=0.5*(focal(l1,y)+focal(l2,y))+CFG["rdrop_alpha"]*rkl(l1,l2)
                else: loss=focal(l1,y)
                loss=loss/_acc
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type,enabled=scaler is not None):
                    la=focal(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")),y)/_acc
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step%_acc==0 or step==len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),CFG["max_grad_norm"])
                (scaler.step(opt),scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        # AFTER
        vm_raw=vmac(); vm,use_ema=vm_raw,False
        if ema is not None and ep>=2:
            ema.apply_shadow(model); vm_ema=vmac()
            if vm_ema>=vm_raw: vm,use_ema=vm_ema,True
            ema.restore(model)
        if vm>best:
            best,noimp=vm,0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(),f"{sd}/best.pt")
            if use_ema: ema.restore(model)
        else: noimp+=1
        if noimp>=CFG["patience"]: break
    model.load_state_dict(torch.load(f"{sd}/best.pt",map_location=device,weights_only=True))
    print(f"  {mk} seed{seed}: best_val_macroF1={best:.4f}")
    return logits_(model,vl), logits_(model,tel)

val_lg,test_lg={},{}
for mk in RUN_MODELS:
    for sd in SEEDS:
        print(f"▶ {mk} seed{sd}")
        v,t=train_model(mk,CFG["models"][mk],sd); val_lg[f"{mk}{sd}"]=v.float(); test_lg[f"{mk}{sd}"]=t.float()
        torch.cuda.empty_cache()

In [ ]:
# ── Ensemble (optimise on val) + final test vs base paper ───────────────────
names=list(val_lg.keys())
def ens(d,w):
    o=None
    for i,n in enumerate(names): o=w[i]*d[n] if o is None else o+w[i]*d[n]
    return o/(np.sum(w)+1e-12)
def opt_w(d,y,k):
    best,bw=1.0,np.ones(k)/k
    for _ in range(40):
        r=minimize(lambda rw:-f1_score(y,ens(d,np.abs(rw)+1e-9).argmax(-1).numpy(),average="macro",zero_division=0),
                   np.random.dirichlet(np.ones(k)),method="Nelder-Mead",options={"maxiter":2000,"xatol":1e-5})
        if r.fun<best: best,bw=r.fun,np.abs(r.x)+1e-9
    return bw/bw.sum()
W=opt_w(val_lg,y_val,len(names))
ens_t=ens(test_lg,W); pr=F.softmax(ens_t,-1).numpy(); yp=ens_t.argmax(-1).numpy(); yt=y_test
pcf=f1_score(yt,yp,average=None,labels=list(range(NC)),zero_division=0)
try: au=roc_auc_score(yt,pr,multi_class="ovr",average="macro",labels=list(range(NC)))
except Exception: au=float("nan")
ours={"macro_f1":round(float(f1_score(yt,yp,average="macro",zero_division=0)),4),
      "weighted_f1":round(float(f1_score(yt,yp,average="weighted",zero_division=0)),4),
      "accuracy":round(float(accuracy_score(yt,yp)),4),"mcc":round(float(matthews_corrcoef(yt,yp)),4),
      "macro_auroc":round(float(au),4),
      "per_class_f1":{classes[i]:round(float(pcf[i]),4) for i in range(NC)}}
print("="*60); print(f"OURS (facebook 15% test, n={len(yt):,})  vs  BASE PAPER"); print("="*60)
print(f"  Macro-F1 : {ours['macro_f1']:.4f}   |  base: {BASE_PAPER['macro_f1']:.4f}   Δ={ours['macro_f1']-BASE_PAPER['macro_f1']:+.4f}")
print(f"  Weighted-F1 {ours['weighted_f1']} | Acc {ours['accuracy']} | MCC {ours['mcc']} | AUROC {ours['macro_auroc']}")
print("\n  Per-class F1 (ours vs base):")
for c in classes:
    b=BASE_PAPER["per_class"].get(c,float("nan")); print(f"    {c:12s} {ours['per_class_f1'][c]:.4f}  vs  {b:.4f}")
print("\n"+classification_report(yt,yp,target_names=classes,zero_division=0))
json.dump({"ours":ours,"base_paper":BASE_PAPER,"weights":{n:float(w) for n,w in zip(names,W)}},
          open(f"{OUT}/comparison.json","w"),indent=2)
print(f"✅ saved {OUT}/comparison.json")